In [ ]:
# 检查 / 打印本 notebook 依赖的关键第三方库版本,便于环境复现或排查依赖问题
from importlib.metadata import version

pkgs = [
    "huggingface_hub",  # to download pretrained weights
    "tokenizers",       # to implement the tokenizer
    "torch",            # to implement the model
]
for p in pkgs:
    print(f"{p} version: {version(p)}")

In [ ]:
# 是否使用 Gemma3 的指令微调(-it)版本模型;True 会在后面加载 google/gemma-3-270m-it 仓库,
# 其对话格式经过微调,更适合配合下面的 chat template 使用
USE_INSTRUCT_MODEL = True

In [ ]:
# ===================== Gemma3 模型定义(含 KV Cache 支持) =====================
# 本单元格实现 Gemma3(以 270M 变体为例)的核心组件:
#   - RMSNorm(均方根归一化,Gemma 系列标准归一化层)
#   - RoPE 旋转位置编码(局部/全局两套不同频率基数)
#   - 分组查询注意力 GroupedQueryAttention(支持 QK-Norm 与 KV Cache)
#   - TransformerBlock(局部滑窗注意力与全局注意力按层交替 + 双重 Post-Norm)
#   - Gemma3Model(词嵌入按 sqrt(emb_dim) 缩放、逐层前向、KV Cache 管理)
# 注:与 Gemma2 不同,Gemma3 官方架构不再使用注意力得分或最终 logits 的 soft-capping(软上限截断),
#    因此下面的实现中也没有 logit soft-capping 相关代码,这是符合官方设计的
import torch
import torch.nn as nn


# FeedForward:门控 MLP,用 GELU(tanh 近似)作为激活的门控分支(类似 SwiGLU 结构)
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        # fc1(门控分支)和 fc2(数值分支)都将输入从 emb_dim 投影到 hidden_dim,二者权重互不共享
        self.fc1 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
        self.fc2 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
        # fc3 将门控相乘后的结果从 hidden_dim 投影回 emb_dim
        self.fc3 = nn.Linear(cfg["hidden_dim"], cfg["emb_dim"], dtype=cfg["dtype"], bias=False)

    # forward: gelu_tanh(fc1(x)) * fc2(x) 得到门控后的隐藏表示,再经 fc3 投影回 emb_dim
    def forward(self, x):
        x_fc1 = self.fc1(x)
        x_fc2 = self.fc2(x)
        x = nn.functional.gelu(x_fc1, approximate="tanh") * x_fc2
        return self.fc3(x)

# ---- RMSNorm:Gemma 系列使用的均方根归一化,不做均值中心化,只按均方根缩放 ----
# 与常见 RMSNorm 实现的两个区别:
#   1) 可学习参数以全 0 初始化,前向时用 (1 + weight) 作为实际缩放系数,便于从零开始训练/微调
#   2) 归一化统计量强制在 float32 下计算,避免 bfloat16 精度不足导致方差估计不准,算完再转回原 dtype
class RMSNorm(nn.Module):
    def __init__(self, emb_dim, eps=1e-6, bias=False):
        super().__init__()
        self.eps = eps
        # Gemma3 stores zero-centered weights and uses (1 + weight) during forward
        self.scale = nn.Parameter(torch.zeros(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim)) if bias else None

    def forward(self, x):
        # Match HF Gemma3: compute norm in float32, then scale by (1 + w)
        input_dtype = x.dtype
        x_f = x.float()
        var = x_f.pow(2).mean(dim=-1, keepdim=True)
        # var 形状: (..., 1),对最后一维(emb_dim 或注意力中的 head_dim)求平方均值,得到方差估计
        x_norm = x_f * torch.rsqrt(var + self.eps)
        # x_norm 与 x 形状相同,只按 RMS 值缩放,不像 LayerNorm 那样先减去均值
        out = x_norm * (1.0 + self.scale.float())
        # 因为 scale 初始化为 0,这里 (1 + scale) 相当于「学习一个相对于恒等变换的增量」

        if self.shift is not None:
            out = out + self.shift.float()

        return out.to(input_dtype)

# ---- RoPE(旋转位置编码)相关工具函数 ----
# Gemma3 对局部滑窗层与全局层使用两套不同的频率基数(theta_base):
#   - 局部层用较小的 rope_local_base(如 10_000),更关注近距离位置关系
#   - 全局层用较大的 rope_base(如 1_000_000),便于建模更长距离的依赖
# 两套 cos/sin 表会在 Gemma3Model.__init__ 中分别预计算好,forward 时按当前层类型选用对应一套
def compute_rope_params(head_dim, theta_base=10_000, context_length=4096, dtype=torch.float32):
    assert head_dim % 2 == 0, "Embedding dimension must be even"

    # Compute the inverse frequencies
    inv_freq = 1.0 / (theta_base ** (torch.arange(0, head_dim, 2, dtype=dtype)[: (head_dim // 2)].float() / head_dim))
    # inv_freq 形状: (head_dim // 2,),每个维度对应一个频率,频率随维度指数衰减(标准 RoPE 频率公式)

    # Generate position indices
    positions = torch.arange(context_length, dtype=dtype)
    # positions 形状: (context_length,),即预先为可能用到的最大长度生成位置索引 0..context_length-1

    # Compute the angles
    angles = positions.unsqueeze(1) * inv_freq.unsqueeze(0)  # Shape: (context_length, head_dim // 2)

    # Expand angles to match the head_dim
    angles = torch.cat([angles, angles], dim=1)  # Shape: (context_length, head_dim)

    # Precompute sine and cosine
    cos = torch.cos(angles)
    sin = torch.sin(angles)
    # cos、sin 形状均为 (context_length, head_dim),后续在 apply_rope 中按 [offset:offset+seq_len] 切片

    return cos, sin


# apply_rope:把预先算好的 cos/sin 应用到 q/k 张量上,实现旋转位置编码
# 采用「旋转一半维度」的写法:把 head_dim 切成前后两半,构造 (-x2, x1) 再与原值加权组合,等价于复数旋转
def apply_rope(x, cos, sin, offset=0):
    # x: (batch_size, num_heads, seq_len, head_dim)
    batch_size, num_heads, seq_len, head_dim = x.shape
    # x 形状为 (batch_size, num_heads, seq_len, head_dim);cos/sin 会按 [offset:offset+seq_len] 切片后广播到该形状
    assert head_dim % 2 == 0, "Head dimension must be even"

    # Split x into first half and second half
    x1 = x[..., : head_dim // 2]  # First half
    x2 = x[..., head_dim // 2 :]  # Second half
    # x1、x2 形状均为 (batch_size, num_heads, seq_len, head_dim // 2)

    # Adjust sin and cos shapes
    cos = cos[offset:offset + seq_len, :].unsqueeze(0).unsqueeze(0)  # Shape: (1, 1, seq_len, head_dim)
    sin = sin[offset:offset + seq_len, :].unsqueeze(0).unsqueeze(0)
    # offset 用于 KV Cache 场景:新 token 的绝对位置从 offset 开始,而不是永远从 0 开始

    # Apply the rotary transformation
    rotated = torch.cat((-x2, x1), dim=-1)
    x_rotated = (x * cos) + (rotated * sin)
    # 展开后等价于 [x1*cos - x2*sin, x2*cos + x1*sin],即标准 RoPE 在实数域的写法

    # It's ok to use lower-precision after applying cos and sin rotation
    return x_rotated.to(dtype=x.dtype)

# ==================== 分组查询注意力(GQA)+ QK-Norm + KV Cache ====================
# 要点:
#   1) GQA: Query 有 num_heads 个头,Key/Value 只有 num_kv_groups 组(num_kv_groups <= num_heads),
#      每组 K/V 被 group_size = num_heads // num_kv_groups 个 Query 头共享,以降低 KV Cache 显存占用
#   2) QK-Norm: 对每个头的 Q、K 分别在 head_dim 维度上做 RMSNorm,提升训练稳定性(Gemma3 新增设计)
#   3) KV Cache: 推理时缓存历史的「未旋转」K、V,每步只需计算新 token 的 Q/K/V,
#      与缓存拼接后统一施加 RoPE,避免重复计算历史 token 的注意力
class GroupedQueryAttention(nn.Module):
    def __init__(
        self, d_in, num_heads, num_kv_groups, head_dim=None, qk_norm=False,
        query_pre_attn_scalar=None, dtype=None,
    ):
        super().__init__()
        assert num_heads % num_kv_groups == 0, "num_heads must be divisible by num_kv_groups"

        # group_size: 每组 KV 被多少个 Query 头共享;例如 num_heads=4, num_kv_groups=1 时 group_size=4(即 Multi-Query Attention)
        self.num_heads = num_heads
        self.num_kv_groups = num_kv_groups
        self.group_size = num_heads // num_kv_groups

        if head_dim is None:
            assert d_in % num_heads == 0, "`d_in` must be divisible by `num_heads` if `head_dim` is not set"
            head_dim = d_in // num_heads

        self.head_dim = head_dim
        self.d_out = num_heads * head_dim

        # W_query 输出维度为 num_heads * head_dim;W_key/W_value 输出维度只有 num_kv_groups * head_dim,远小于前者(GQA 的显存优势来源)
        self.W_query = nn.Linear(d_in, self.d_out, bias=False, dtype=dtype)
        self.W_key = nn.Linear(d_in, num_kv_groups * head_dim, bias=False, dtype=dtype)
        self.W_value = nn.Linear(d_in, num_kv_groups * head_dim, bias=False, dtype=dtype)

        self.out_proj = nn.Linear(self.d_out, d_in, bias=False, dtype=dtype)

        # 若启用 qk_norm(Gemma3 默认开启),对每个头的 head_dim 做 RMSNorm;否则不做归一化
        if qk_norm:
            self.q_norm = RMSNorm(head_dim, eps=1e-6)
            self.k_norm = RMSNorm(head_dim, eps=1e-6)
        else:
            self.q_norm = self.k_norm = None

        # Gemma3 用固定的 query_pre_attn_scalar(而非 head_dim)计算注意力缩放系数,
        # 该值来自官方配置,不随 head_dim 变化,便于解耦「头维度」与「缩放系数」两个超参
        if query_pre_attn_scalar is not None:
            self.scaling = (query_pre_attn_scalar) ** -0.5
        else:
            self.scaling = (head_dim) ** -0.5

    # forward 参数说明:
    #   mask: 本层使用的注意力掩码(局部滑窗或全局,由 TransformerBlock 决定)
    #   cos, sin: 对应局部或全局的 RoPE 预计算表
    #   start_pos: 本次前向新增 token 的起始绝对位置(用于 RoPE 与 mask 对齐)
    #   cache: 该层的 (k, v) KV Cache 元组,首次调用为 None
    def forward(self, x, mask, cos, sin, start_pos=0, cache=None):
        b, num_tokens, _ = x.shape
        # x 形状: (b, num_tokens, d_in);num_tokens 是本次前向新增的 token 数,可能是整段 prompt,也可能是逐 token 生成时的 1

        # Apply projections
        queries = self.W_query(x)  # (b, num_tokens, num_heads * head_dim)
        keys = self.W_key(x)       # (b, num_tokens, num_kv_groups * head_dim)
        values = self.W_value(x)   # (b, num_tokens, num_kv_groups * head_dim)
        # 此时 queries/keys/values 还未拆分成多头,形状分别为 (b, num_tokens, num_heads*head_dim) 与 (b, num_tokens, num_kv_groups*head_dim)

        # Reshape
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        keys_new = keys.view(b, num_tokens, self.num_kv_groups, self.head_dim).transpose(1, 2)
        values_new = values.view(b, num_tokens, self.num_kv_groups, self.head_dim).transpose(1, 2)
        # 拆分多头后: queries 形状 (b, num_heads, num_tokens, head_dim);keys_new/values_new 形状 (b, num_kv_groups, num_tokens, head_dim)

        # Optional Q/K normalization (applied to raw tensors)
        if self.q_norm:
            queries = self.q_norm(queries)
        if self.k_norm:
            keys_new = self.k_norm(keys_new)
        # 注意执行顺序: 线性投影 -> reshape 成多头 -> QK-Norm(对未旋转的 Q/K 做)-> RoPE 旋转

        # Keep unrotated in cache; rotate after concatenation
        # 中文说明:KV Cache 中始终保存「未做 RoPE 旋转」的 K、V;等把历史与新 K 拼接完之后再统一施加 RoPE,
        # 这样同一个 token 在不同的生成步里,其缓存内容保持不变,不需要对旧缓存做增量修正
        prev_len = 0
        if cache is not None:
            prev_k, prev_v = cache  # cached as unrotated
            if prev_k is not None:
                prev_len = prev_k.size(2)
                # prev_len: 已缓存的历史 token 数;keys_new/values_new 只是本次新增部分,形状 (b, num_kv_groups, num_tokens, head_dim)
                keys_cat_raw = torch.cat([prev_k, keys_new], dim=2)      # unrotated
                values_cat_raw = torch.cat([prev_v, values_new], dim=2)  # raw V
                # keys_cat_raw/values_cat_raw 形状: (b, num_kv_groups, prev_len+num_tokens, head_dim),即「历史 + 新增」拼接后的完整 K/V
            else:
                keys_cat_raw = keys_new
                values_cat_raw = values_new
        else:
            keys_cat_raw = keys_new
            values_cat_raw = values_new

        # RoPE: queries at absolute start_pos; keys with offset corrected by prev_len
        queries = apply_rope(queries, cos, sin, offset=start_pos)
        keys = apply_rope(keys_cat_raw, cos, sin, offset=start_pos - prev_len)
        # 对 queries 用绝对位置 start_pos;对拼接后的 keys 用 (start_pos - prev_len) 作为起始偏移,
        # 这样可以让 keys 中「历史部分」仍旧对应它们各自写入缓存时的绝对位置,新拼接部分对应到 start_pos 之后的位置

        # Scale queries
        queries = queries * self.scaling
        # 在 RoPE 之后再做缩放,不影响相对位置编码本身,只整体改变注意力得分的幅度

        # Update cache with unrotated keys and unscaled raw values
        if cache is not None and cache[0] is not None:
            next_cache = (
                torch.cat([cache[0], keys_new], dim=2),
                torch.cat([cache[1], values_new], dim=2),
            )
        else:
            next_cache = (keys_new, values_new)
        # next_cache 保存的是「历史 + 本次新增」拼接后的完整未旋转 K 和原始 V,交给外层写回该层的 KV Cache

        # Expand K and V to match number of heads
        keys = keys.repeat_interleave(self.group_size, dim=1)
        values = values_cat_raw.repeat_interleave(self.group_size, dim=1)
        # 用 repeat_interleave 把 K/V 的组数广播到与 Query 相同的头数(GQA 的核心操作),
        # 广播后 keys/values 形状变为 (b, num_heads, kv_len, head_dim),可以与 queries 逐头做点积

        # Attention
        attn_scores = queries @ keys.transpose(2, 3)
        # attn_scores 形状: (b, num_heads, num_tokens, kv_len),kv_len = prev_len + num_tokens,即含全部历史缓存
        attn_scores = attn_scores.masked_fill(mask, -torch.inf)
        # mask 中为 True 的位置表示「不可见」(未来位置,或超出滑窗范围),用 -inf 屏蔽后再做 softmax
        attn_weights = torch.softmax(attn_scores, dim=-1)

        context = (attn_weights @ values).transpose(1, 2).reshape(b, num_tokens, self.d_out)
        # 注意力输出先是 (b, num_heads, num_tokens, head_dim),transpose + reshape 后变回 (b, num_tokens, d_out)
        out = self.out_proj(context)

        return out, next_cache

# ==================== TransformerBlock:局部/全局注意力交替 + Gemma3 的双重 Post-Norm 结构 ====================
# 每一层要么是 'sliding_attention'(局部滑窗注意力),要么是 'full_attention'(全局注意力),
# 具体的交替模式由外部配置 cfg['layer_types'] 指定
# 归一化结构为 Pre-Norm 与 Post-Norm 结合(区别于 GPT 系列只用 Pre-Norm):
#   input_layernorm -> Attention -> post_attention_layernorm -> 残差相加
#   pre_feedforward_layernorm -> FeedForward -> post_feedforward_layernorm -> 残差相加
class TransformerBlock(nn.Module):

    def __init__(self, cfg, attn_type):
        super().__init__()
        self.attn_type = attn_type
        # sliding_window: 局部注意力层能看到的历史 token 数上限,超出部分会被滑窗掩码屏蔽,并从 KV Cache 中物理丢弃
        self.sliding_window = cfg["sliding_window"]

        self.att = GroupedQueryAttention(
            d_in=cfg["emb_dim"],
            num_heads=cfg["n_heads"],
            num_kv_groups=cfg["n_kv_groups"],
            head_dim=cfg["head_dim"],
            qk_norm=cfg["qk_norm"],
            query_pre_attn_scalar=cfg["query_pre_attn_scalar"],
            dtype=cfg["dtype"],
        )
        self.ff = FeedForward(cfg)
        self.input_layernorm = RMSNorm(cfg["emb_dim"], eps=1e-6)
        self.post_attention_layernorm = RMSNorm(cfg["emb_dim"], eps=1e-6)
        self.pre_feedforward_layernorm = RMSNorm(cfg["emb_dim"], eps=1e-6)
        self.post_feedforward_layernorm = RMSNorm(cfg["emb_dim"], eps=1e-6)

    # forward 收到的 mask_global/mask_local、cos_global/sin_global、cos_local/sin_local 都是针对「完整位置区间」预先算好的张量,
    # 本层根据 self.attn_type 选择其中一套使用
    def forward(
        self,
        x,
        mask_global,
        mask_local,
        cos_global,
        sin_global,
        cos_local,
        sin_local,
        start_pos=0,
        cache=None
    ):
        # Shortcut connection for attention block
        shortcut = x
        x = self.input_layernorm(x)

        # 根据本层类型选择:局部滑窗层用 mask_local + 局部 RoPE 频率;全局层用 mask_global + 全局 RoPE 频率
        if self.attn_type == "sliding_attention":
            if cache is not None and isinstance(cache, tuple):
                prev_k, _ = cache
                eff_kv_len = prev_k.size(2) + x.size(1)
                # eff_kv_len = 该层 KV Cache 中「历史 token 数」+「本次新增 token 数」,
                # 即真正参与注意力计算的 K 长度(滑窗层的历史部分已被截断到最多 sliding_window)
            else:
                eff_kv_len = x.size(1)
                # 首次前向(还没有 cache)时,eff_kv_len 就等于当前输入的 token 数
            # Take the last `eff_kv_len` columns so mask width equals K length
            attn_mask = mask_local[..., -eff_kv_len:]
            # 风险提示:mask_local 是针对「完整位置区间」[0, pos_end) 预先构造好的滑窗掩码,
            # 这里通过截取最后 eff_kv_len 列,使掩码宽度与该层物理 KV Cache 的长度对齐;
            # 若某层 KV Cache 的截断逻辑与这里的切片逻辑不一致,可能导致掩码列与实际 K 的相对位置错位。
            # 原实现逻辑保持不变,这里只做风险标注,未做修改。
            cos = cos_local
            sin = sin_local
        else:
            attn_mask = mask_global
            cos = cos_global
            sin = sin_global

        x_attn, next_cache = self.att(x, attn_mask, cos, sin, start_pos=start_pos, cache=cache)
        # 滑窗层专属:注意力算完后,如果该层 KV Cache 长度超过 sliding_window,则只保留最近 sliding_window 个 token,
        # 让局部层的显存占用恒定,不随生成长度无限增长
        if next_cache is not None and self.attn_type == "sliding_attention":
            k, v = next_cache
            if k.size(2) > self.sliding_window:
                k = k[:, :, -self.sliding_window:, :]
                v = v[:, :, -self.sliding_window:, :]
            next_cache = (k, v)

        x_attn = self.post_attention_layernorm(x_attn)
        x = shortcut + x_attn

        # Shortcut connection for feed forward block
        shortcut = x
        x_ffn = self.pre_feedforward_layernorm(x)
        x_ffn = self.ff(x_ffn)
        x_ffn = self.post_feedforward_layernorm(x_ffn)
        x = shortcut + x_ffn
        return x, next_cache

# ==================== Gemma3Model:整体模型(词嵌入缩放 + 逐层前向 + KV Cache 管理) ====================
class Gemma3Model(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg["layer_types"] is not None and len(cfg["layer_types"]) == cfg["n_layers"]

        # Main model parameters
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"], dtype=cfg["dtype"])

        self.blocks = nn.ModuleList([
            TransformerBlock(cfg, attn_type) for attn_type in cfg["layer_types"]
        ])

        self.final_norm = RMSNorm(cfg["emb_dim"], eps=1e-6)
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False, dtype=cfg["dtype"])
        self.cfg = cfg
        self.current_pos = 0  # Track current position in KV cache

        # Reusable utilities
        # 局部层 RoPE 表:使用较小的 rope_local_base,适合滑窗内的短距离位置编码
        cos_local, sin_local = compute_rope_params(
            head_dim=cfg["head_dim"],
            theta_base=cfg["rope_local_base"],
            context_length=cfg["context_length"],
            dtype=torch.float32,
        )
        # 全局层 RoPE 表:使用较大的 rope_base,适合建模超出滑窗范围的长距离依赖
        cos_global, sin_global = compute_rope_params(
            head_dim=cfg["head_dim"],
            theta_base=cfg["rope_base"],
            context_length=cfg["context_length"],
            dtype=torch.float32,
        )
        self.register_buffer("cos_local", cos_local, persistent=False)
        self.register_buffer("sin_local", sin_local, persistent=False)
        self.register_buffer("cos_global", cos_global, persistent=False)
        self.register_buffer("sin_global", sin_global, persistent=False)

    # _create_masks: 一次性构造「全局注意力掩码」和「局部滑窗注意力掩码」,两者都基于 [0, pos_end) 的完整位置区间;
    # 使用 KV Cache 时,通过 pos_start/pos_end 只截取当前 step 需要的「query 行」,但「key 列」仍覆盖到 pos_end(即全部历史 key)
    def _create_masks(self, cur_len, device, pos_start=0, pos_end=None):
        if pos_end is None:
            pos_end = cur_len
        total_len = pos_end

        ones = torch.ones((total_len, total_len), dtype=torch.bool, device=device)
        # ones/掩码矩阵均为 (total_len, total_len) 的方阵,total_len = pos_end,即到目前为止出现过的最大绝对位置

        # mask_global_full (future is masked: j > i)
        #     j:  0 1 2 3 4 5 6 7
        #  i
        #     0:  0 1 1 1 1 1 1 1
        #     1:  0 0 1 1 1 1 1 1
        #     2:  0 0 0 1 1 1 1 1
        #     3:  0 0 0 0 1 1 1 1
        #     4:  0 0 0 0 0 1 1 1
        #     5:  0 0 0 0 0 0 1 1
        #     6:  0 0 0 0 0 0 0 1
        #     7:  0 0 0 0 0 0 0 0
        mask_global_full = torch.triu(ones, diagonal=1)

        # far_past (too far back is masked: i - j >= sliding_window)
        # where sliding_window = 4
        #     j:  0 1 2 3 4 5 6 7
        #  i
        #     0:  0 0 0 0 0 0 0 0
        #     1:  0 0 0 0 0 0 0 0
        #     2:  0 0 0 0 0 0 0 0
        #     3:  0 0 0 0 0 0 0 0
        #     4:  1 0 0 0 0 0 0 0
        #     5:  1 1 0 0 0 0 0 0
        #     6:  1 1 1 0 0 0 0 0
        #     7:  1 1 1 1 0 0 0 0
        far_past_full = torch.triu(ones, diagonal=self.cfg["sliding_window"]).T

        # Local (sliding_window) = future OR far-past
        # mask_local
        #     j:  0 1 2 3 4 5 6 7
        # i
        # 0:      0 1 1 1 1 1 1 1
        # 1:      0 0 1 1 1 1 1 1
        # 2:      0 0 0 1 1 1 1 1
        # 3:      0 0 0 0 1 1 1 1
        # 4:      1 0 0 0 0 1 1 1
        # 5:      1 1 0 0 0 0 1 1
        # 6:      1 1 1 0 0 0 0 1
        # 7:      1 1 1 1 0 0 0 0
        mask_local_full = mask_global_full | far_past_full

        row_slice = slice(pos_start, pos_end)
        mask_global = mask_global_full[row_slice, :pos_end][None, None, :, :]
        mask_local = mask_local_full[row_slice,  :pos_end][None, None, :, :]
        # 最终返回的 mask_global / mask_local 形状均为 (1, 1, pos_end-pos_start, pos_end),
        # 前两个维度是为了方便广播到 (batch, num_heads, ...) 形状的注意力得分张量
        return mask_global, mask_local


    # Gemma3Model.forward: 单次前向,可选地读写 KV Cache
    #   - cache 为 None: 常规一次性前向(如一次性计算整段 prompt 的 logits)
    #   - cache 不为 None: 增量推理模式,每次只传入「新增」的 token(可以是整段 prompt 的首次填充,也可以是单个 token)
    def forward(self, input_ids, cache=None):
        b, seq_len = input_ids.shape
        # input_ids 形状: (b, seq_len);seq_len 是本次调用新增的 token 数,而不是累计的总长度
        x = self.tok_emb(input_ids) * (self.cfg["emb_dim"] ** 0.5)
        # 关键点:Gemma 系列在词嵌入之后要乘以 sqrt(emb_dim) 做缩放,
        # 这样可以让嵌入向量的方差与其他层激活值处于同一量级,是 Gemma 系列的标志性设计之一

        if cache is not None:
            pos_start = self.current_pos
            pos_end = pos_start + seq_len
            self.current_pos = pos_end
            # pos_start/pos_end: 本次新增 token 在「整条序列」里的绝对位置区间 [pos_start, pos_end)
            # current_pos 记录到目前为止已经处理过的 token 总数,供下一次调用作为新的 pos_start
            mask_global, mask_local = self._create_masks(
                cur_len=seq_len, device=x.device, pos_start=pos_start, pos_end=pos_end
            )
        else:
            pos_start = 0
            # 不使用 KV Cache 时,相当于把整段序列当作「从头开始」一次性处理
            mask_global, mask_local = self._create_masks(
                cur_len=seq_len, device=x.device, pos_start=0, pos_end=seq_len
            )

        for i, block in enumerate(self.blocks):
            blk_cache = cache.get(i) if cache is not None else None
            # blk_cache: 第 i 层的 (k, v) KV Cache,首次调用为 None
            x, new_blk_cache = block(
                x,
                mask_global=mask_global,
                mask_local=mask_local,
                cos_global=self.cos_global,
                sin_global=self.sin_global,
                cos_local=self.cos_local,
                sin_local=self.sin_local,
                start_pos=pos_start,  # position of first new token
                cache=blk_cache,
            )

            # 将该层更新后的 KV Cache 写回,供下一次增量前向复用
            if cache is not None:
                cache.update(i, new_blk_cache)

        # Final layernorm + projection
        x = self.final_norm(x)
        logits = self.out_head(x.to(self.cfg["dtype"]))
        # logits 形状: (b, seq_len, vocab_size);投影前先把归一化输出转换回配置的 dtype(如 bfloat16),与 out_head 权重 dtype 对齐
        return logits

    # reset_kv_cache: 开始新一轮生成前调用,把「已处理 token 计数」清零,避免复用上一次生成遗留的位置信息
    def reset_kv_cache(self):
        self.current_pos = 0



# 【bug 修复说明】原始 notebook 在此处混入了一行纯文本 "2. Initialize model"(本应是独立的 markdown 标题 cell),
# 直接留在代码 cell 里会被当作 Python 语句解析,导致 ast.parse / 执行本单元格时报 SyntaxError,现予以删除,
# 仅保留下面这行注释,标记这里是「初始化模型」逻辑的起点
# 2. 初始化模型:定义 Gemma3-270M 的配置字典,并实例化 Gemma3Model
GEMMA3_CONFIG_270M = {
    # 词表大小(Gemma3 词表远大于 GPT-2,覆盖多语言)
    "vocab_size": 262_144,
    # 官方支持的最大上下文长度
    "context_length": 32_768,
    # 隐藏层(embedding)维度
    "emb_dim": 640,
    # 注意力头总数(Query 头数)
    "n_heads": 4,
    # Transformer 层数
    "n_layers": 18,
    # FeedForward 中间隐藏维度
    "hidden_dim": 2048,
    # 每个注意力头的维度;注意 head_dim=256 而 emb_dim/n_heads=640/4=160,说明 head_dim 是独立配置的,
    # 并不等于 emb_dim // n_heads,这也是为什么 GroupedQueryAttention 允许显式传入 head_dim 参数
    "head_dim": 256,
    # 是否启用 QK-Norm(Gemma3 的关键设计之一,对 Q/K 做 RMSNorm 以提升训练稳定性)
    "qk_norm": True,
    # KV 组数;这里为 1,即所有 Query 头共享同一组 K/V(等价于 Multi-Query Attention,最省 KV Cache 显存)
    "n_kv_groups": 1,
    # 局部滑窗层使用的 RoPE 频率基数(较小)
    "rope_local_base": 10_000.0,
    # 全局层使用的 RoPE 频率基数(较大)
    "rope_base": 1_000_000.0,
    # 局部滑窗注意力的窗口大小(每个滑窗层只能看到最近 512 个 token)
    "sliding_window": 512,
    # 逐层指定注意力类型:大多数层是局部滑窗注意力,每 6 层里有 1 层是全局注意力,
    # 这种「局部为主、周期性全局」的设计能在几乎不损失长距离建模能力的前提下大幅降低 KV Cache 显存占用
      "layer_types": [
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention"
    ],
    # 模型权重与激活的计算精度
    "dtype": torch.bfloat16,
    # 计算注意力缩放系数时使用的固定值(替代 head_dim),对应 GroupedQueryAttention.__init__ 中的 scaling = value ** -0.5
    "query_pre_attn_scalar": 256,
}
# 注:该配置字典中没有 attn_logit_softcapping / final_logit_softcapping 字段——
# Gemma3(不同于 Gemma2)已经不再对注意力得分或最终 logits 做 soft-capping(软上限截断),这是符合官方架构设计的

# 固定随机种子后实例化模型(此时权重仍是随机初始化的,后面会用 HuggingFace 预训练权重覆盖)
torch.manual_seed(123)
model = Gemma3Model(GEMMA3_CONFIG_270M)
# 最后一行是裸表达式 `model`,在 Notebook 中会打印出模型结构(逐层子模块列表)
model

In [ ]:
# 用一个长度为 3 的示例 token 序列做一次前向传播(不使用 KV Cache),用来快速验证模型结构搭建是否正确
# 输入 unsqueeze(0) 后形状为 (batch_size=1, seq_len=3);输出 logits 形状为 (1, 3, vocab_size)
model(torch.tensor([1, 2, 3]).unsqueeze(0))

In [ ]:
# 统计模型的总参数量
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

# 若输出层 out_head 与词嵌入 tok_emb 权重共享(weight tying),会被重复计数一次,
# 这里减去词嵌入参数量,得到「去重后」的真实参数量
# Account for weight tying
total_params_normalized = total_params - model.tok_emb.weight.numel()
print(f"\nTotal number of unique parameters: {total_params_normalized:,}")

In [ ]:
# 估算模型在给定 dtype 下的显存/内存占用(仅参数 + 梯度 + buffer,不包含激活值和优化器状态)
def calc_model_memory_size(model, input_dtype=torch.float32):
    total_params = 0
    total_grads = 0
    for param in model.parameters():
        # Calculate total number of elements per parameter
        param_size = param.numel()
        total_params += param_size
        # Check if gradients are stored for this parameter
        if param.requires_grad:
            total_grads += param_size

    # Calculate buffer size (non-parameters that require memory)
    # buffers 包含 register_buffer 注册的 cos_local/sin_local/cos_global/sin_global 等非训练张量
    total_buffers = sum(buf.numel() for buf in model.buffers())

    # Size in bytes = (Number of elements) * (Size of each element in bytes)
    # We assume parameters and gradients are stored in the same type as input dtype
    element_size = torch.tensor(0, dtype=input_dtype).element_size()
    total_memory_bytes = (total_params + total_grads + total_buffers) * element_size

    # Convert bytes to gigabytes
    total_memory_gb = total_memory_bytes / (1024**3)

    return total_memory_gb

# 对比 float32 与 bfloat16 两种精度下的理论显存占用(bfloat16 约为 float32 的一半)
print(f"float32 (PyTorch default): {calc_model_memory_size(model, input_dtype=torch.float32):.2f} GB")
print(f"bfloat16: {calc_model_memory_size(model, input_dtype=torch.bfloat16):.2f} GB")

In [ ]:
# 按优先级选择可用的计算设备:CUDA > Apple Silicon(MPS) > CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

model.to(device);

4. Load pretrained weights

In [ ]:
# ==================== 将 HuggingFace 格式的 Gemma3 预训练权重加载进自定义模型 ====================
# HF safetensors 里的参数名遵循 `model.layers.{l}.xxx` 的命名规范,这里逐一映射到本 notebook 自定义的模块属性上
def load_weights_into_gemma(model, param_config, params):

    # assign: 校验形状一致后,把右侧权重张量拷贝进左侧参数,同时保持左侧原有的 dtype/device 不变
    def assign(left, right, tensor_name="unknown"):
        if left.shape != right.shape:
            raise ValueError(f"Shape mismatch in tensor '{tensor_name}'. Left: {left.shape}, Right: {right.shape}")

        with torch.no_grad():
            if isinstance(right, torch.Tensor):
                left.copy_(right)
            else:
                left.copy_(torch.as_tensor(right, dtype=left.dtype, device=left.device))

        return left

    # Embedding weights
    # 词嵌入权重,形状为 (vocab_size, emb_dim)
    if "model.embed_tokens.weight" in params:
        model.tok_emb.weight = assign(
            model.tok_emb.weight,
            params["model.embed_tokens.weight"],
            "model.embed_tokens.weight",
        )

    # Iterate over transformer layers
    # 逐层拷贝:注意力投影、QK-Norm、FeedForward(SwiGLU 的三个线性层)以及四个 RMSNorm 的权重
    for l in range(param_config["n_layers"]):
        block = model.blocks[l]
        att = block.att
        # Attention projections
        att.W_query.weight = assign(
            att.W_query.weight,
            params[f"model.layers.{l}.self_attn.q_proj.weight"],
            f"model.layers.{l}.self_attn.q_proj.weight",
        )
        att.W_key.weight = assign(
            att.W_key.weight,
            params[f"model.layers.{l}.self_attn.k_proj.weight"],
            f"model.layers.{l}.self_attn.k_proj.weight",
        )
        att.W_value.weight = assign(
            att.W_value.weight,
            params[f"model.layers.{l}.self_attn.v_proj.weight"],
            f"model.layers.{l}.self_attn.v_proj.weight",
        )
        att.out_proj.weight = assign(
            att.out_proj.weight,
            params[f"model.layers.{l}.self_attn.o_proj.weight"],
            f"model.layers.{l}.self_attn.o_proj.weight",
        )
        # QK normalization weights
        # q_norm/k_norm 的 scale 形状均为 (head_dim,),对应前面 RMSNorm 类中「零初始化 + (1+w)」缩放参数
        att.q_norm.scale = assign(
            att.q_norm.scale,
            params[f"model.layers.{l}.self_attn.q_norm.weight"],
            f"model.layers.{l}.self_attn.q_norm.weight",
        )
        att.k_norm.scale = assign(
            att.k_norm.scale,
            params[f"model.layers.{l}.self_attn.k_norm.weight"],
            f"model.layers.{l}.self_attn.k_norm.weight",
        )
        # Feed forward weights
        # HF 命名与本实现的对应关系: gate_proj -> fc1(门控分支),up_proj -> fc2(数值分支),down_proj -> fc3(下投影)
        block.ff.fc1.weight = assign(
            block.ff.fc1.weight,
            params[f"model.layers.{l}.mlp.gate_proj.weight"],
            f"model.layers.{l}.mlp.gate_proj.weight",
        )
        block.ff.fc2.weight = assign(
            block.ff.fc2.weight,
            params[f"model.layers.{l}.mlp.up_proj.weight"],
            f"model.layers.{l}.mlp.up_proj.weight",
        )
        block.ff.fc3.weight = assign(
            block.ff.fc3.weight,
            params[f"model.layers.{l}.mlp.down_proj.weight"],
            f"model.layers.{l}.mlp.down_proj.weight",
        )
        # LayerNorm weights
        block.input_layernorm.scale = assign(
            block.input_layernorm.scale,
            params[f"model.layers.{l}.input_layernorm.weight"],
            f"model.layers.{l}.input_layernorm.weight",
        )
        block.post_attention_layernorm.scale = assign(
            block.post_attention_layernorm.scale,
            params[f"model.layers.{l}.post_attention_layernorm.weight"],
            f"model.layers.{l}.post_attention_layernorm.weight",
        )
        # Pre‑ and post‑feed forward norms
        # 部分 Gemma3 checkpoint 可能不包含 pre/post_feedforward_layernorm 权重,这里用 in 判断做兼容处理
        pre_key = f"model.layers.{l}.pre_feedforward_layernorm.weight"
        post_key = f"model.layers.{l}.post_feedforward_layernorm.weight"
        if pre_key in params:
            block.pre_feedforward_layernorm.scale = assign(
                block.pre_feedforward_layernorm.scale,
                params[pre_key],
                pre_key,
            )
        if post_key in params:
            block.post_feedforward_layernorm.scale = assign(
                block.post_feedforward_layernorm.scale,
                params[post_key],
                post_key,
            )

    # Final LayerNorm
    # 模型最终输出前的归一化层
    if "model.norm.weight" in params:
        model.final_norm.scale = assign(
            model.final_norm.scale,
            params["model.norm.weight"],
            "model.norm.weight",
        )
    # Output head
    # 若 checkpoint 中没有单独的 lm_head.weight,说明该模型使用「权重绑定」(weight tying),
    # 直接让输出层复用词嵌入矩阵,而不是报错
    if "lm_head.weight" in params:
        model.out_head.weight = assign(
            model.out_head.weight,
            params["lm_head.weight"],
            "lm_head.weight",
        )
    else:
        model.out_head.weight = model.tok_emb.weight
        print("Model uses weight tying.")

# ==================== KVCache:按层存储的简单 K/V 缓存容器 ====================
# self.cache 是长度为 n_layers 的列表;每个元素要么是 None(尚未缓存),要么是一个 (k, v) 元组,
# 其中 k、v 的形状均为 (batch, num_kv_groups, cached_len, head_dim)(滑窗层的 cached_len 不会超过 sliding_window)
class KVCache:
    def __init__(self, n_layers):
        self.cache = [None] * n_layers

    def get(self, layer_idx):
        return self.cache[layer_idx]

    def update(self, layer_idx, value):
        self.cache[layer_idx] = value

    def get_all(self):
        return self.cache

    # reset: 开始新一轮生成(新的 prompt/新的 batch)前调用,清空所有层的缓存
    def reset(self):
        for i in range(len(self.cache)):
            self.cache[i] = None

In [ ]:
# Uncomment and run the following code if you are executing the notebook for the first time
# 如果是第一次运行本 notebook,需要先登录 HuggingFace(部分 Gemma 模型为受限访问,需接受许可协议)

#from huggingface_hub import login
#login()
import json
import os
from pathlib import Path
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download, snapshot_download

# 选择要下载的 Gemma3 模型规模;这里固定为 270M,即该 notebook 使用的最小档位
CHOOSE_MODEL = "270m"

# 根据前面 cell 中设置的 USE_INSTRUCT_MODEL,决定加载「基础版」还是「指令微调(-it)版」仓库
if USE_INSTRUCT_MODEL:
    repo_id = f"google/gemma-3-{CHOOSE_MODEL}-it"
else:
    repo_id = f"google/gemma-3-{CHOOSE_MODEL}"


local_dir = Path(repo_id).parts[-1]

# 270M 模型体积小,权重打包在单个 safetensors 文件里,直接下载该文件即可
if CHOOSE_MODEL == "270m":
    weights_file = hf_hub_download(
        repo_id=repo_id,
        filename="model.safetensors",
        local_dir=local_dir,
    )
    weights_dict = load_file(weights_file)
# 更大规格的模型权重会被切分成多个 shard 文件,需要先读取 index.json 里的 weight_map,
# 再逐个下载对应 shard 并合并成一个完整的权重字典
else:
    repo_dir = snapshot_download(repo_id=repo_id, local_dir=local_dir)
    index_path = os.path.join(repo_dir, "model.safetensors.index.json")
    with open(index_path, "r") as f:
        index = json.load(f)

    weights_dict = {}
    for filename in set(index["weight_map"].values()):
        shard_path = os.path.join(repo_dir, filename)
        shard = load_file(shard_path)
        weights_dict.update(shard)

# 把下载好的权重字典加载进已经实例化的模型,并转移到目标计算设备;随后释放原始权重字典以节省内存
load_weights_into_gemma(model, GEMMA3_CONFIG_270M, weights_dict)
model.to(device)
del weights_dict

3. Load tokenizer

In [ ]:
from tokenizers import Tokenizer


# ==================== GemmaTokenizer:基于 tokenizers 库封装的 Gemma 分词器 ====================
# 直接加载 HuggingFace 发布的 tokenizer.json(不依赖 transformers 的 AutoTokenizer),
# 保持本 notebook「standalone,不依赖 transformers」的定位
class GemmaTokenizer:
    def __init__(self, tokenizer_file_path: str):
        tok_file = Path(tokenizer_file_path)
        self._tok = Tokenizer.from_file(str(tok_file))

        # Gemma 对话格式中用到的一组特殊 token 字符串
        self.bos_token = "<bos>"
        self.eos_token = "<eos>"
        self.pad_token = "<pad>"
        self.start_of_turn_token = "<start_of_turn>"
        self.end_of_turn_token = "<end_of_turn>"

        # 把上面的特殊 token 字符串转换为对应的 token id,方便后续拼接 / 判断
        self.bos_token_id = self._tok.token_to_id(self.bos_token)
        self.eos_token_id = self._tok.token_to_id(self.eos_token)
        self.pad_token_id = self._tok.token_to_id(self.pad_token)
        self.start_of_turn_token_id = self._tok.token_to_id(self.start_of_turn_token)
        self.end_of_turn_token_id = self._tok.token_to_id(self.end_of_turn_token)

        # 编码时是否自动添加 <bos>/<eos>,以及是否清理分词产生的多余空格
        self.add_bos_token = True
        self.add_eos_token = False
        self.clean_up_tokenization_spaces = False

    # encode: 文本 -> token id 列表
    def encode(self, text: str, add_special_tokens: bool = True) -> list[int]:
        return self._tok.encode(text, add_special_tokens=add_special_tokens).ids

    # decode: token id(或 id 列表)-> 文本;skip_special_tokens 控制是否过滤掉 <bos>/<eos> 等特殊 token
    def decode(self, ids: list[int] | int, skip_special_tokens: bool = False) -> str:
        if isinstance(ids, int):
            ids = [ids]
        return self._tok.decode(ids, skip_special_tokens=skip_special_tokens)

    # apply_chat_template: 按 Gemma 的对话格式拼接多轮消息,格式形如:
    #   <start_of_turn>user\n...内容...<end_of_turn>\n<start_of_turn>model\n...
    # 注意:OpenAI 风格里的 'assistant' 角色,在 Gemma 的格式里要写成 'model'
    def apply_chat_template(self, messages, tokenize=False, add_generation_prompt=False):
        text = ""
        for message in messages:
            role = message["role"]
            if role == "assistant":
                role = "model"
            content = message["content"]
            text += f"{self.start_of_turn_token}{role}\n{content}{self.end_of_turn_token}\n"

        if add_generation_prompt:
            text += f"{self.start_of_turn_token}model\n"

        if tokenize:
            return self.encode(text)
        return text


# 模块级的便捷函数:对单轮用户输入套用 Gemma 的 chat template,并在末尾追加生成提示(model 角色起始标记)
def apply_chat_template(user_text):
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": user_text}],
        tokenize=False,
        add_generation_prompt=True,
    )

# 优先复用前面下载权重时已经落盘的 tokenizer.json;如果本地没有,再单独从 HF Hub 下载
tokenizer_file_path = os.path.join(local_dir, "tokenizer.json")
if not os.path.exists(tokenizer_file_path):
    try:
        tokenizer_file_path = hf_hub_download(repo_id=repo_id, filename="tokenizer.json", local_dir=local_dir)
    except Exception as e:
        print(f"Warning: failed to download tokenizer.json: {e}")
        tokenizer_file_path = "tokenizer.json"


# 实例化分词器
tokenizer = GemmaTokenizer(tokenizer_file_path=tokenizer_file_path)

In [ ]:
# 构造一个示例用户提问,并套用 Gemma 对话模板(自动加上 <start_of_turn>/<end_of_turn> 等特殊标记)
prompt = "Give me a short introduction to large language models."
prompt = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}],
    tokenize=False,
    add_generation_prompt=True,
)



# 将套用模板后的完整文本编码为 token id 列表
input_token_ids = tokenizer.encode(prompt)
# 再解码回文本,用于肉眼检查模板拼接、特殊 token 是否符合预期
text = tokenizer.decode(input_token_ids)
text

5. Generate text

In [ ]:
# Optionally use torch.compile for an extra speed-up
# model = torch.compile(model)

# ==================== 基于 KV Cache 的流式文本生成 ====================
# 与不带 KV Cache 的朴素生成相比,这里每一步只把「新产生的 1 个 token」喂给模型,
# 历史上下文的注意力信息由每一层内部的 KV Cache 承担,避免了重复计算整段历史的开销
def generate_text_basic_stream(model, token_ids, max_new_tokens, eos_token_id=None, context_size=None):
    model.eval()

    with torch.no_grad():
        # 为每一层创建一个空的 KV Cache 容器,并把模型内部记录的位置计数器清零
        cache = KVCache(n_layers=model.cfg["n_layers"])
        model.reset_kv_cache()

        # Prime the cache with the initial context
        # 用完整的 prompt 做一次前向,把 prompt 对应的 K/V 写入每一层的缓存(即「预热」/「填充」KV Cache),
        # 同时得到 prompt 最后一个位置的 logits,用于生成第一个新 token
        logits = model(token_ids, cache=cache)

        for _ in range(max_new_tokens):
            next_token = torch.argmax(logits[:, -1], dim=-1, keepdim=True)
            # 贪心解码:直接取概率最大的 token 作为下一个 token(未使用采样/温度等策略)

            if eos_token_id is not None and torch.all(next_token == eos_token_id):
                break

            yield next_token

            token_ids = torch.cat([token_ids, next_token], dim=1)
            # 这里的 token_ids 只用于函数外部记录已生成的完整序列,并不会整段重新喂给模型

            # Feed only the new token to the model; cache handles history
            logits = model(next_token, cache=cache)
            # 关键点:只把新生成的 1 个 token 传给模型(形状 (b, 1)),模型内部会自动把它与各层 KV Cache 拼接后计算注意力

# 把编码好的 prompt token id 转成模型输入张量,形状为 (batch_size=1, prompt_len)
input_token_ids_tensor = torch.tensor(input_token_ids, device=device).unsqueeze(0)


if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()


# 逐 token 调用生成器,一边生成一边把新 token 解码成文本并打印出来(流式输出效果)
for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=500,
    eos_token_id=tokenizer.end_of_turn_token_id
):
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )


# 生成结束后,如果在 CUDA 设备上运行,打印本次生成过程中峰值显存占用,便于评估 KV Cache 的显存开销
if torch.cuda.is_available():
    def calc_gpu_gb(x):
        return f"{x / 1024 / 1024 / 1024:.2f} GB"

    print(f"\n\nGPU memory used: {calc_gpu_gb(torch.cuda.max_memory_allocated())}")